# 02 - Assessing differential abundance

In [ ]:
%autosave 0

## Load libraries

In [ ]:
import os
import sys
from pathlib import Path

# Set the base project directory
base_proj_dir = Path("/projects/site/pred/ihb-g-deco/USERS/adaml9/intestine_fate/notebooks/tf_ko_panel_analysis")

# Append necessary paths for module imports
sys.path.append("/projects/site/pred/ihb-g-deco/USERS/adaml9/bioinfo-blueprint/src/python")
sys.path.append(str(base_proj_dir))

import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import pertpy as pt
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from dynaconf import Dynaconf
from tqdm import tqdm

# Load settings
settings = Dynaconf(
    settings_files=[base_proj_dir / 'settings.toml', base_proj_dir / '.secrets.toml'],
)

sc.settings.figdir = settings.ANALYSIS.figures_dir

In [ ]:
sc.__version__

In [ ]:
data_dir = Path(settings.IO.combined_data) / "all-sample" / "DGE_filtered"
anndata_dir = Path(settings.IO.anndata)

## Utility functions

In [ ]:
def group_by_max(expr: pd.DataFrame) -> list[str]:
    """
    Order genes by the column (group) where they show the maximum expression.
    Within each group, genes are sorted by their maximum expression (descending).
    """
    max_group = expr.idxmax(axis=1)
    max_value = expr.max(axis=1)
    ordered = []

    for group in expr.columns:
        group_genes = max_group[max_group == group].index
        sorted_genes = max_value[group_genes].sort_values(ascending=False).index
        ordered.extend(sorted_genes)

    return list(ordered)

## Utility paremeters

In [ ]:
# Specify color palette for each comparison
comparison_palette = {
    "non_eecs_annotation": {
        "Stem Cells": "#1f77b4",          # deep blue
        "TA Cells": "#4c91c6",            # medium blue

        # absorptive lineage
        "Enterocytes": "#2ca02c",         # green

        # secretory lineage
        "Goblet Cells": "#ff7f0e",        # orange
    },
    "eecs_annotation": {
        # EEC progenitors
        "EEC Progenitors": "#9467bd",   # purple
        
        # EEC subtypes 
        "EC Cells": "#d62728",            # strong red
        "X Cells": "#e377c2",             # magenta
        "D Cells": "#17becf",             # cyan / turquoise
        "I/N Cells": "#7f7f7f",           # dark grey
        "K Cells": "#bcbd22",             # olive
    },
    "eecs_milestones_pred": {
        # Pure populations
        "NGN3+ cells": "#9467bd",   # purple (progenitors)
        "EC/X/D/I/K cells": "#fdae61",    # warm orange blend
        "EC cells": "#d62728",      # strong red
        "X/D/I/K cells": "#c2a5cf",       # lavender blend
        "X cells": "#e377c2",       # magenta
        #"D/I/K cells": "#7fbf7b",         # green-cyan blend
        "D cells": "#17becf",       # cyan
        "I/K cells": "#bcbd22",     # olive
    },
    "eecs_annotation_pred": {
        # EEC progenitors
        "EEC Progenitors": "#9467bd",   # purple

        # EEC subtypes 
        "EC Cells": "#d62728",            # strong red
        "X Cells": "#e377c2",             # magenta
        "D Cells": "#17becf",             # cyan / turquoise
        "I/N Cells": "#7f7f7f",           # dark grey
        "K Cells": "#bcbd22",             # olive
    }
}

# Minimum number of cells per cell type to be included in the analysis
min_cells = 50

# Set fdr threshold
est_fdr = 0.4

## Run differential abundance analysis for specific comparisons

In [ ]:
# Specify input list of adata objects
input_dict = {
    "all_eecs": {
        "path": anndata_dir / "tf_ko_panel_contrastiveVI_annotated_eecs_milestones_including_cystic_v3.h5ad",
        "test_annotations": ["milestones_pred"],
        "prefix": "eecs",
    },
    #"all_eecs": {
    #    "path": anndata_dir / "tf_ko_panel_contrastiveVI_annotated_eecs_milestones_including_cystic_v3.h5ad",
    #    "test_annotations": ["annotation", "annotation_pred", "milestones_pred"],
    #    "prefix": "eecs",
    # },
    #"all_non_eecs": {
    #    "path": anndata_dir / "tf_ko_panel_contrastiveVI_raw_annotated_non_eec.h5ad",
    #    "test_annotations": ["annotation"],
    #    "prefix": "non_eecs",
    # }
}

In [ ]:
# --- Helper: generate a 4-digit RNG key ---
def generate_rng_key():
    return int(''.join(np.random.choice(list('0123456789'), size=4, replace=False)))

# --- Run scCODA with automatic re-run if effects explode ---
def run_sccoda_with_retry(model, data, tf, est_fdr=0.05, max_retries=10):

    modality = f"coda_{tf}"
    rng_key = generate_rng_key()

    for attempt in range(max_retries):

        # Run model
        model.run_nuts(
            data,
            modality_key=modality,
            num_warmup=200,
            num_samples=1000,
            rng_key=rng_key
        )
        model.set_fdr(data, modality_key=modality, est_fdr=est_fdr)

        # Get effects
        effects_df = model.get_effect_df(data, modality_key=modality)

        # Check for extreme log2fc
        if (effects_df["log2-fold change"].abs() < 10).all():
            return effects_df, rng_key  # success

        # Otherwise retry with a new RNG key
        print(f"[{tf}] Extreme log2-FC detected (rng={rng_key}). Retrying...")
        rng_key = generate_rng_key()

    raise RuntimeError(f"scCODA failed to converge for TF '{tf}' after {max_retries} attempts.")

In [ ]:
def process_input(input_name, input_data):
    
    adata = sc.read(input_data["path"])
    test_annotations = input_data["test_annotations"]
    prefix = input_data["prefix"]
    output_dir = Path(settings.ANALYSIS.tables_dir) / input_name 
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Filter conditions based on minimum cell count per cell type
    valid_conditions = []
    invalid_conditions = []
    for condition, group in adata.obs.groupby("condition"):
        if (group["annotation"].value_counts() >= min_cells).any():
            valid_conditions.append(condition)
        else:
            invalid_conditions.append(condition)
            
    print(invalid_conditions)

    adata = adata[adata.obs["condition"].isin(valid_conditions)].copy()
    tfs = [x for x in adata.obs["condition"].unique() if x != "Control"]
    print(f"Processing {input_name} with {len(tfs)} TF KOs...")
    
    for annotation in test_annotations:
        data = (
            adata.obs[["condition", annotation]]
            .groupby(["condition", annotation])
            .size()
            .unstack(fill_value=0)
            .T
        )
        data_cell_count = data.sum(axis=0).rename("total_cells")
        data_cell_count.to_csv(output_dir / f"{prefix}_{annotation}_cell_counts_per_condition.csv")
        print("Saved cell counts per condition.")
        
        data_cell_prop = data / data.sum(axis=0)
        data_cell_prop.to_csv(output_dir / f"{prefix}_{annotation}_cell_prop_per_condition.csv")
        print("Saved cell proportions per condition.")
        
        # In addition save cell counts and proportions for the controls
        data = (
            adata.obs[adata.obs["condition"] == "Control"]
            .groupby(["sample", annotation])
            .size()
            .unstack(fill_value=0)
            .T
        )
        data_control_cell_count = data.sum(axis=0).rename("total_cells")
        data_control_cell_count.to_csv(output_dir / f"{prefix}_{annotation}_control_cell_counts_per_sample.csv")
        print("Saved control cell counts per sample.")
        
        data_control_cell_prop = data / data.sum(axis=0)
        data_control_cell_prop.to_csv(output_dir / f"{prefix}_{annotation}_control_cell_prop_per_sample.csv")
        print("Saved control cell proportions per sample.")
        
        # Define cell type identifier based on annotation
        cell_type_identifier = annotation

        sccoda_model = pt.tl.Sccoda()
        sccoda_data = sccoda_model.load(
            adata,
            type="cell_level",
            generate_sample_level=True,
            cell_type_identifier=cell_type_identifier,
            sample_identifier="sample",
            covariate_obs=["condition"],
        )
    
        for tf_select in tfs:
            print(f"Processing TF: {tf_select}")
            out_path = output_dir / f"sccoda_effects_{prefix}_{annotation}" / f"sccoda_effects_{tf_select}.csv"
            out_path.parent.mkdir(parents=True, exist_ok=True)
            
            sccoda_data.mod[f"coda_{tf_select}"] = sccoda_data["coda"][
                sccoda_data["coda"].obs["condition"].isin(["Control", tf_select])
            ].copy()

            cond = sccoda_data[f"coda_{tf_select}"].obs["condition"].astype("category")
            cond = cond.cat.reorder_categories(["Control", tf_select], ordered=True)
            sccoda_data[f"coda_{tf_select}"].obs["condition"] = cond
            
            if not os.path.exists(out_path.parent / f"sccoda_boxplots_{tf_select}.pdf"):
                print("Generating boxplot figure...")
                fig = sccoda_model.plot_boxplots(
                    sccoda_data,
                    modality_key=f"coda_{tf_select}",
                    feature_name="condition",
                    figsize=(12, 5),
                    add_dots=True,
                    return_fig=True
                )
                fig.savefig(out_path.parent / f"sccoda_boxplots_{tf_select}.pdf")
                plt.close()
            
            if os.path.exists(out_path):
                print(f"Results for {tf_select} already exist at {out_path}. Skipping...")
                continue
            
            adata_subset = adata[adata.obs["condition"].isin(["Control", tf_select])].copy()

            # compute per-sample proportions for Control
            control_props = (
                pd.crosstab(
                    adata_subset.obs.loc[adata_subset.obs["condition"] == "Control", "sample"],
                    adata_subset.obs.loc[adata_subset.obs["condition"] == "Control", cell_type_identifier],
                    normalize="index"
                )
                .T.median(axis=1)
            )

            ko_props = (
                pd.crosstab(
                    adata_subset.obs.loc[adata_subset.obs["condition"] != "Control", "sample"],
                    adata_subset.obs.loc[adata_subset.obs["condition"] != "Control", cell_type_identifier],
                    normalize="index"
                )
                .T.median(axis=1)
            )

            df = pd.DataFrame({
                "Control": control_props,
                "TF_KO": ko_props
            }).fillna(0)

            df = df.reset_index().rename(columns={tf_select: "TF_KO", cell_type_identifier: "Cell Type"})
            df["log2_fc"] = np.log2((df["TF_KO"] + 1e-5) / (df["Control"] + 1e-5))
            df["abs_log2_fc"] = df["log2_fc"].abs()

            ref_celltype = df.sort_values(by="abs_log2_fc")["Cell Type"].iloc[0]

            sccoda_data = sccoda_model.prepare(
                sccoda_data,
                modality_key=f"coda_{tf_select}",
                formula="condition",
                reference_cell_type=ref_celltype,
            )
            
            effects_df, rng_key = run_sccoda_with_retry(
                model=sccoda_model,
                data=sccoda_data,
                tf=tf_select,
                est_fdr=est_fdr
            )
            
            effects_df = pd.DataFrame(effects_df).reset_index()
            effects_df = effects_df.merge(df, on="Cell Type")
            effects_df["ref_celltype"] = ref_celltype
            effects_df["rng_key"] = rng_key
            effects_df["fdr"] = est_fdr
            effects_df.to_csv(out_path, index=False)

In [ ]:
from pathlib import Path
import submitit

log_dir = Path("/projects/site/pred/ihb-g-deco/USERS/adaml9/submitit_logs")
log_dir.mkdir(exist_ok=True)

# Create executor
executor = submitit.AutoExecutor(folder=log_dir)

executor.update_parameters(
    timeout_min=1 * 60,         # 1 hour
    slurm_partition="batch_cpu",
    qos="3h",
    cpus_per_task=1,
    mem_gb=200,
)

jobs = {}
for input_name, input_data in input_dict.items():
    job = executor.submit(process_input, input_name, input_data)
    jobs[input_name] = job
    print(f"Submitted {input_name} → Job ID {job.job_id}")

## Load abundance results

In [ ]:
import numpy as np
import pandas as pd
import marsilea as ma
from marsilea.plotter import ColorMesh, Labels, Colors, StackBar, Numbers

In [ ]:
for input_name, input_data in input_dict.items():
    test_annotations = input_data["test_annotations"]
    prefix = input_data["prefix"]
    output_dir = Path(settings.ANALYSIS.tables_dir) / input_name 
    
    # Load all comparisons / test_annotations
    for annotation in test_annotations:
        sccoda_df = pd.concat([pd.read_csv(x) for x in list((output_dir / f"sccoda_effects_{prefix}_{annotation}").glob("sccoda_effects_*.csv"))], axis=0)
        sccoda_df["condition"] = sccoda_df["Covariate"].apply(lambda x: x.replace("conditionT.", ""))
        print(sccoda_df["condition"].unique())
        # Remove BAMBI from condition
        sccoda_df = sccoda_df[sccoda_df["condition"] != "BAMBI"]
        sccoda_df["comparison"] = annotation
        sccoda_df["sig"] = ~(
            sccoda_df.groupby(["condition", "comparison"])["log2-fold change"]
            .transform(lambda x: (x == 0).all())
        ) 
    
        anno_df = sccoda_df[sccoda_df["comparison"] == annotation][["condition", "sig"]].drop_duplicates().sort_values(by="condition")
        sccoda_mat = pd.pivot_table(
            sccoda_df[sccoda_df["comparison"] == annotation],
            index="Cell Type",
            columns="condition",
            values="log2_fc" #log2_fc, log2-fold change
        )
        print(sccoda_mat.shape)
        if sccoda_mat.empty:
            print("No data in sccoda matrix. Skipping visualization.")
            continue
        
        plot_palette = comparison_palette[f"{prefix}_{annotation}"]
     
        # Parse sccoda matrix as lfc dataframe
        cell_lfc = pd.DataFrame(sccoda_mat).loc[plot_palette.keys()]
        sorted_cols = group_by_max(
                cell_lfc.T
            )
        cell_lfc = cell_lfc[sorted_cols]
 
        # Load cell counts and proportions
        cell_count = pd.read_csv(output_dir / f"{prefix}_{annotation}_cell_counts_per_condition.csv", index_col=0).loc[sorted_cols]
        cell_prop = pd.read_csv(output_dir / f"{prefix}_{annotation}_cell_prop_per_condition.csv", index_col=0)[sorted_cols].loc[plot_palette.keys()]

        # Reorder annotation dataframe
        anno_df = anno_df.set_index("condition").loc[sorted_cols]
        
        # Load control cell counts and proportions
        data_control_cell_count = pd.read_csv(output_dir / f"{prefix}_{annotation}_control_cell_counts_per_sample.csv", index_col=0)
        data_control_cell_prop = pd.read_csv(output_dir / f"{prefix}_{annotation}_control_cell_prop_per_sample.csv", index_col=0)
        data_control_cell_prop = data_control_cell_prop.dropna(axis=1)
        data_control_cell_count = data_control_cell_count.loc[list(data_control_cell_prop.columns)]

        cell_count = pd.concat([cell_count, data_control_cell_count], axis=0)
        cell_prop = pd.concat([cell_prop, data_control_cell_prop], axis=1)
                
        missing_cols = set(cell_prop.columns) - set(cell_lfc.columns)
        for col in missing_cols:
            cell_lfc[col] = 0.0
            anno_df.loc[col] = False
        
        h = ma.ClusterBoard(cell_lfc, width=15, height=1.5)
        h.add_layer(ColorMesh(cell_lfc, cmap="RdBu_r", center=0, vmin=-2, vmax=2, label="Abundance\nlog2FC"))
        h.add_top(StackBar(cell_prop, label="Cell Type", colors=plot_palette), pad=0.1, size=1)
        h.add_top(Numbers(np.squeeze(cell_count.values, axis=1), label="Cell Count", color="#525252", show_value=False), pad=0.1, size=1)
        h.add_left(Labels(cell_lfc.index), size=1, pad=0.1)
        h.add_bottom(Colors(anno_df["sig"], palette=["#525252", "#d9d9d9"], label="Sig"), size=0.2, pad=0.1)
        #h.group_cols(anno_df["sig"].values)
        h.add_bottom(Labels(cell_lfc.columns), size=1, pad=0.1)
        h.add_legends()
        
        with plt.rc_context({'axes.grid': False}):
            h.render()
            plt.savefig(output_dir / f"sccoda_effects_{prefix}_{annotation}_heatmap_joined.pdf", bbox_inches='tight')

## Exports

In [ ]:
from pathlib import Path
import pandas as pd


def load_sccoda(effects_dir: str, annotation: str = "annotation"):
    """
    Load and process SCCODA effect tables for budding_eecs or cystic_eecs.

    Parameters
    ----------
    output_dir : str
        Base directory for the output tables.
    input_name : str
        Name of the input dataset.
    prefix : str
        Either "budding_eecs" or "cystic_eecs".
    annotation : str, optional
        Annotation string used in file naming, by default "annotation".

    Returns
    -------
    pd.DataFrame
        Processed SCCODA results with added 'condition', 'comparison', and 'sig' columns.
    """    

    # Collect all CSV files
    files = list(effects_dir.glob("sccoda_effects_*.csv"))
    if not files:
        raise FileNotFoundError(f"No SCCODA CSV files found in {effects_dir}")

    # Load & concatenate
    df = pd.concat([pd.read_csv(f) for f in files], axis=0)

    # Extract clean condition name
    df["condition"] = df["Covariate"].str.replace("conditionT.", "", regex=False)

    # Remove BAMBI
    df = df[df["condition"] != "BAMBI"]

    # Tag comparison group
    df["comparison"] = annotation

    # Significant if ANY log2FC ≠ 0
    df["sig"] = ~(
        df.groupby(["condition", "comparison"])["log2-fold change"]
        .transform(lambda x: (x == 0).all())
    )

    return df

In [ ]:
annotation = "annotation"

datasets = {
    "eecs": ("all_eecs_pre", "sccoda_effects_eecs_annotation"),
    "non_eecs": ("all_non_eecs", "sccoda_effects_non_eecs_annotation"),
}

dfs = []

for label, (dataset_dir, effects_subdir) in datasets.items():
    output_dir = Path(settings.ANALYSIS.tables_dir) / dataset_dir
    effects_dir = output_dir / effects_subdir

    df = load_sccoda(
        effects_dir=effects_dir,
        annotation=annotation,
    )

    # Optional: keep track of source dataset
    df["group"] = label

    dfs.append(df)

# Combine into one dataframe
sccoda_df = pd.concat(dfs, axis=0)

In [ ]:
n_sig_df = sccoda_df[["sig", "condition", "group"]].drop_duplicates().query("sig == True")

pd.crosstab(n_sig_df["condition"], n_sig_df["group"]).sum(axis=0)

In [ ]:
df = sccoda_df.query("sig == True")[["Cell Type", "condition", "log2-fold change"]]
len(df["condition"].unique())

In [ ]:
df

In [ ]:
# Export scCoda summary table
df.to_csv(Path(settings.ANALYSIS.tables_dir) / "tf_ko_panel_contrastiveVI_da_summary_v3.tsv", sep="\t", index=False)

In [ ]:
annotation = "milestones_pred"

datasets = {
    "all_eecs": ("all_eecs", "sccoda_effects_eecs_milestones_pred")
}

dfs = []

for label, (dataset_dir, effects_subdir) in datasets.items():
    output_dir = Path(settings.ANALYSIS.tables_dir) / dataset_dir
    effects_dir = output_dir / effects_subdir

    df = load_sccoda(
        effects_dir=effects_dir,
        annotation=annotation,
    )

    # Optional: keep track of source dataset
    df["group"] = label

    dfs.append(df)

# Combine into one dataframe
sccoda_df = pd.concat(dfs, axis=0)

In [ ]:
n_sig_df = sccoda_df[["sig", "condition", "group"]].drop_duplicates().query("sig == True")

pd.crosstab(n_sig_df["condition"], n_sig_df["group"]).sum(axis=0)

In [ ]:
sccoda_df

In [ ]:
anno_df_eecs = sccoda_df.query('group == "all_eecs"')[["condition", "sig"]].drop_duplicates().sort_values(by="condition")
sccoda_mat_eecs = pd.pivot_table(sccoda_df.query('group == "all_eecs"'), index="Cell Type", columns="condition", values="log2_fc")

In [ ]:
def group_by_max(expr: pd.DataFrame) -> list[str]:
    """
    Order genes by the column (group) where they show the maximum expression.
    Within each group, genes are sorted by their maximum expression (descending).
    """
    max_group = expr.idxmax(axis=1)
    max_value = expr.max(axis=1)
    ordered = []

    for group in expr.columns:
        group_genes = max_group[max_group == group].index
        sorted_genes = max_value[group_genes].sort_values(ascending=False).index
        ordered.extend(sorted_genes)

    return list(ordered)

In [ ]:
ordering_ct = ["NGN3+ cells", "EC/X/D/I/K cells", "EC cells", "X/D/I/K cells", "X cells", "D cells", "I/K cells"]

In [ ]:
pd.DataFrame(sccoda_mat_eecs)

In [ ]:
cell_lfc = pd.DataFrame(sccoda_mat_eecs).loc[ordering_ct]

sorted_cols = group_by_max(
    cell_lfc.T
)
cell_lfc = cell_lfc[sorted_cols]
anno_df_eecs["condition"] = anno_df_eecs["condition"].astype("category").cat.reorder_categories(cell_lfc.columns)
anno_df_eecs = anno_df_eecs.sort_values(by="condition")

In [ ]:
# Export cell_lfc of significant effects
significant_effects_eecs = cell_lfc[list(anno_df_eecs.query('sig == True')['condition'].values)]
significant_effects_eecs.to_csv(Path(settings.ANALYSIS.tables_dir) / "sccoda_tf_ko_significant_effects_eecs_milestones_pred_v3.csv")